# QAOA：把四个点分成两组

MaxCut 要把图中的点分成两组，让尽可能多的边跨过两组。用 0、1 表示分组，每个量子比特对应一个点。
本例是一张四边形图，最优划分可以切开全部四条边。

我们只训练两步，观察概率分布与期望切边数的变化。所有计算和可视化只使用 qutai 与 Python 标准库。

## 和 VQE 有什么区别？

两者都用经典优化器调整电路参数，再通过量子态的能量期望来衡量结果。

| 项目 | VQE | 本例 QAOA |
| --- | --- | --- |
| 目标 | 降低给定哈密顿量的能量 | 降低表示组合优化问题的成本 |
| 电路结构 | 根据问题选择参数化电路 | 固定交替执行成本演化、混合演化 |
| 参数 | 取决于所选电路 | 每层两个参数 gamma、beta |
| 本例结果 | 量子态与能量 | 分组位串、切边数及其概率 |

QAOA 可以理解为采用特定电路结构的变分优化；它与 VQE 共用反向传播和 SGD。


In [1]:
from qutai import nn, visualization
from qutai.quantum import QAOA, maxcut

device = "cuda"
dtype = "float32"


## 1. 用普通列表描述图

每个二元组表示一条无向边。下面四条边围成一个四边形。
如果一条边的两个端点分到不同组，这条边就被“切开”。


In [2]:
qubits = 4
edges = [(0, 1), (1, 2), (2, 3), (3, 0)]

visualization.Graph(range(qubits), edges, title="四个点、四条边的 MaxCut 问题")

## 2. 把切边数变成要最小化的能量

`maxcut(edges)` 为每条边构造成本 $(Z_iZ_j-I)/2$：

- 两端分组相同，能量为 0。
- 两端分组不同，能量为 −1。

所以 **成本能量 = −切边数**，最小化能量就对应最大化切边数。

这里只把 16 种分组列出来核对理论答案；这份枚举用于小例子检验，不属于 QAOA 算法本身。
q0 对应位串最左边一位，例如 `0101` 表示点 0、2 在第一组，点 1、3 在第二组。


In [3]:
cost = maxcut(edges)
bitstrings = [format(index, f"0{qubits}b") for index in range(2 ** qubits)]
cut_values = [sum(bits[first] != bits[second] for first, second in edges) for bits in bitstrings]
maximum = max(cut_values)
optimal_states = [bits for bits, cut in zip(bitstrings, cut_values) if cut == maximum]

print("理论最大切边数:", maximum)
print("最优分组:", optimal_states)

visualization.Table(
    [[bits, cut, -cut] for bits, cut in zip(bitstrings, cut_values)],
    columns=["分组位串", "切边数", "成本能量"],
    title="16 种分组的实际得分",
)

理论最大切边数: 4
最优分组: ['0101', '1010']


## 3. 建立 QAOA 电路

电路先对每个比特施加 H 门，得到等幅叠加态。随后每层依次执行：

1. **成本演化** $e^{-i\gamma C}$：根据分组成本改变振幅的相位。
2. **混合演化** $e^{-i\beta\sum X}$：用 RX 门改变不同分组之间的振幅分布。

本例深度为 1，只有两个可训练参数。`weight` 的每一行是 `[gamma, beta]`。
seed 只为复现示例显式指定，平时省略即可；随机初始化并不保证初始分布优于均匀分布。

ZZ 演化由 CNOT → RZ → CNOT 实现，混合演化使用 RX(2 beta)。这些角度中的倍数来自 RX/RZ 的半角定义。


In [4]:
algorithm = QAOA(qubits, cost, depth=1, seed=42).to(device, dtype=dtype)
optimizer = nn.SGD(algorithm, learning_rate=0.1)

print("每层的 [gamma, beta]:", algorithm.weight.tolist())
print("可训练参数数量:", algorithm.weight.size)

visualization.Circuit(algorithm.circuit(), title="QAOA：H → 成本演化 → X 混合演化")

每层的 [gamma, beta]: [[0.8760443925857544, -2.984445333480835]]
可训练参数数量: 2


## 4. 先看初始期望值

每次测量得到的切边数都是整数，而“期望切边数”是按概率加权的平均值，可以是小数。
例如某些测量切开 2 条边、另一些切开 4 条边，平均值就可能是 2.9。
四个点的均匀分布对应期望切边数 2；随机初始化的 QAOA 参数可能让它变高，也可能变低。


In [5]:
initial_energy = algorithm.expectation().item()
history = [[0, initial_energy, -initial_energy]]

print("训练前成本能量:", initial_energy)
print("训练前期望切边数:", -initial_energy)

训练前成本能量: -1.421631932258606
训练前期望切边数: 1.421631932258606


## 5. 严格更新两次

成本期望本身就是优化目标，直接调用 `energy.backward()`。
每次前向都会用当前 gamma、beta 重建对应的门角度，已有反向引擎会累加共享参数在各个门中的贡献。

两步只验证训练链路，不保证收敛或一定找到最优解。


In [6]:
for step in range(1, 3):
    optimizer.zero_grad()
    energy = algorithm.expectation()
    energy.backward()
    optimizer.step()

    current_energy = algorithm.expectation().item()
    history.append([step, current_energy, -current_energy])
    print(f"step {step}  energy {current_energy:.8f}  expected cut {-current_energy:.8f}")

visualization.Table(history, columns=["更新次数", "成本能量", "期望切边数"], title="两次更新的结果")

step 1  energy -2.58572841  expected cut 2.58572841
step 2  energy -2.93521976  expected cut 2.93521976


In [7]:
visualization.Plot(title="QAOA 的两步训练", xlabel="参数更新次数", ylabel="期望切边数").line(
    [row[0] for row in history], [row[2] for row in history], label="当前期望切边数"
).line([0, 2], [maximum, maximum], label="理论最大切边数")

## 6. 看训练后的分组概率

`algorithm.run()` 得到当前参数下的量子态。概率分布告诉我们各个位串出现的可能性。


In [8]:
state = algorithm.run()
probabilities = state.probabilities()

visualization.Probabilities(state, title="更新两次后的分组概率")

从这个状态独立采样 1000 次，再把出现次数、理论概率和实际切边数放在一起对照。
本例固定采样 seed，便于复现；去掉 seed 就会自然随机。采样不会改变原量子态。


In [9]:
counts = state.counts(1000, seed=42)
results = sorted(zip(bitstrings, cut_values, probabilities), key=lambda row: row[2], reverse=True)

visualization.Table(
    [[bits, cut, f"{probability:.2%}", counts.get(bits, 0)] for bits, cut, probability in results],
    columns=["分组位串", "切边数", "理论概率", "1000 次采样中的次数"],
    title="分组结果与实际得分",
)

## 7. 把最可能的分组画出来

两种颜色表示两组，绿色边表示跨组、已经被切开的边。最可能的位串只是按当前理论概率选出的候选分组。
这个四边形有两个对称的最优答案：`0101` 和 `1010`，它们只是互换了两组的名称。


In [10]:
best = max(bitstrings, key=lambda bits: probabilities[int(bits, 2)])
best_cut = cut_values[int(best, 2)]
best_partition = {wire: int(best[wire]) for wire in range(qubits)}

print("当前最可能的分组:", best)
print("这个分组实际切开的边数:", best_cut)
print("平均切边数:", sum(cut * probability for cut, probability in zip(cut_values, probabilities)))

visualization.Graph(range(qubits), edges, groups=best_partition, title=f"候选分组 {best}：切开 {best_cut} 条边")

当前最可能的分组: 0101
这个分组实际切开的边数: 4
平均切边数: 2.9352193474769592


这里的 QAOA 支持固定 I/Z 乘积的实数加权和，使用全体 X 混合器。
本例用 MaxCut 展示流程，同样的电路模板也可以接收手写的 Z、ZZ 或更长 Z 字符串成本。

参考：[QAOA 原始论文](https://arxiv.org/abs/1411.4028)、[PennyLane 的 MaxCut 成本定义](https://docs.pennylane.ai/en/stable/code/api/pennylane.qaoa.cost.maxcut.html)。
